## 5.2 损失函数 - 多分类交叉熵 （Cross Entropy Loss）


#### 1. 什么是多分类问题？
多分类问题指的是：
`一个样本只能属于 多个类别中的一个。`

例如：手写数字识别

`0 1 2 3 4 5 6 7 8 9`

共有 10 个类别。

不同于机器学习中不同的分类算法，在NN中：
* 输出层神经元 = 10
* 每个神经元对应一个数字类别

模型的任务是：

`从 10 个类别中选择概率最大的一个。`

##### 1.1 模型输出是什么？
在手写数字识别中，假设模型输出：

`[2.1, 0.3, -1.2, 1.5, 0.4, -0.6, 0.8, -0.3, 0.2, -1.0]`

这些值叫做：

`logits（未归一化得分）`

⚠️注意：
* 这些 不是概率。
* 还没有经过 Softmax 激活

##### 1.2 为什么需要 Softmax？
为了把 logits 转换为概率，我们需要使用：`Softmax`

Softmax 会把输出变成：

`[0.42, 0.06, 0.01, 0.22, 0.07, 0.02, 0.12, 0.02, 0.05, 0.01]`

这些值可以理解为：

每个数字的预测概率

满足：
* 每个值 ∈ (0,1)
* 总和 = 1

因此：`Softmax 输出可以解释为 每个类别的概率。`

#### 2. 交叉熵公式
多分类交叉熵的公式：

`Loss = -SUM_C(y_i * log(p_i))`

其中：
* C → 类别数量
* y_i → 真实标签（one-hot）
* p_i → 模型预测概率

##### 2.1 为什么公式是这样？
因为：
`y = one-hot`

例如：

[0,0,1,0]

所以只有正确类别的部分会留下：

此时公式为：

`Loss = -log(p_correct)`

因此：

交叉熵本质就是：

`Loss = -log(正确类别的预测概率)`

##### 2.2 为什么要加 log？
log 的作用是：

`让错误预测受到更大的惩罚。`

例如：
```
概率	loss
0.9	0.105
0.5	0.693
0.1	2.302
0.01	4.605
```
可以看到：

如果正确类别概率很小：

loss 会非常大

这会迫使模型快速修正错误。

#### 3. 交叉熵的核心思想与计算流程
交叉熵的核心思想其实非常简单：

`如果模型预测的概率越接近真实标签，损失就越小。`

如果预测完全正确：

`loss → 接近 0`

如果预测完全错误：

`loss → 很大`

##### 3.1 进行 One-Hot 编码
在分类问题中，真实标签通常只是一个 类别编号：

比如，真实class：

`3`

但是在神经网络公式计算中，单个标签无法与输出层的多个神经元结果进行计算

所以我们需要把标签表示为一个 向量：

`[0,0,0,1,0,0,0,0,0,0,0]`

因此引入了：

`One-Hot 编码`

##### 3.2 什么是 One-Hot？
One-Hot 向量的规则：
* 正确类别位置 = 1
* 其他位置 = 0

如果类别数量是 10，并且真实类别是 3：

`[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]`

含义是：

`P(true class) = 第4个位置`

##### 3.3 为什么需要 One-Hot？
① 统一向量形式

神经网络所有计算都是 向量计算。

单个真实标签无法与输出层的多个神经元结果进行计算

例如：
* 预测概率 = [p0, p1, p2, p3, ...]
* 真实标签 = [y0, y1, y2, y3, ...]

这样可以统一用矩阵运算计算 loss。

② 方便数学计算

Cross Entropy 的公式是：

`Loss = -SUM_C(y_i * log(p_i))`

其中：
* y_i → 真实标签（one-hot）
* p_i → 模型预测概率

如果使用 one-hot：

`y = [0,0,0,1,0,0,0,0,0,0]`

计算时：

只有正确类别会留下

因为：

`0 × anything = 0`

最后公式就变成：

`Loss = -log(p_correct)`

也就是：

只关注正确类别的概率。

##### 3.4 一个简单例子
假设任务是：识别手写数字

class类别为：

`0 1 2 3 4 5 6 7 8 9`

因此：
* 输出层神经元 = 10
* 每个神经元对应一个数字类别

得到模型的预测概率

`[0.05, 0.08, 0.02, 0.70, 0.04, 0.02, 0.03, 0.03, 0.02, 0.01]`

对应 真实类别 3 的 one-hot 编码：

`[0,0,0,1,0,0,0,0,0,0,0]`

loss：

`Loss = -log(0.70)`

因此：
* 如果模型预测 0.95 → loss 很小
* 如果模型预测 0.10 → loss 很大

#### 4. Cross Entropy + Softmax 的关系

##### 4.1 注意点1:
在 PyTorch 中：

`CrossEntropyLoss`

已经包含了 Softmax。

也就是说：

`CrossEntropyLoss = LogSoftmax + NLLLoss`

因此：

⚠️ 模型输出 logits，不要自己加 Softmax。

❌错误写法

`Linear → Softmax → CrossEntropyLoss`

✅ 正确写法

`Linear → CrossEntropyLoss`

CrossEntropyLoss 会自动处理 Softmax。

##### 4.2 注意点2:
虽然理论公式使用 one-hot，但在 PyTorch 中并不需要自己写 one-hot。

例如：

`target = 3`

PyTorch 的：

`CrossEntropyLoss`

会自动帮我们处理。

所以在 PyTorch 中：

`target = [3,1,0,2]`

而不是：
```
[0,0,0,1,0,0,0,0,0,0]
[0,1,0,0,0,0,0,0,0,0]
[1,0,0,0,0,0,0,0,0,0]
[0,0,1,0,0,0,0,0,0,0]
```

#### 5. PyTorch 实现 Cross Entropy

##### 5.1 定义损失函数
`criterion = nn.CrossEntropyLoss()`

##### 5.2 输入格式
CrossEntropyLoss 需要两个输入：

`loss = criterion(pred, target)`

pred（模型输出）

形状：
```
X: (batch_size， 输入特征个数）
W: (输出神经元个数，输入特征个数)
```
得到：

`(batch_size , 输出神经元个数)`

例如：

`(32, 10)`

表示：
* 32 个样本
* 10 个类别

target（真实标签）

形状：

`(batch_size)`

例如：

[3,1,0,2,4]

⚠️注意：
* pytorch自动处理 One-Hot 编码

##### 5.3 完整模型 + 训练

In [1]:
import torch
import torch.nn as nn

# 1. 定义模型
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_1 = nn.Linear(10, 5)  # 输入层到隐藏层
        self.relu = nn.ReLU()  # 激活函数
        self.layer_2 = nn.Linear(5,4) # 隐藏层到输出层
    
    def forward(self, x):
        x = self.layer_1(x)  # 输入数据通过第一层
        x = self.relu(x)  # 激活函数
        x = self.layer_2(x)  # 通过第二层得到输出
        return x
    
# 2. 准备数据
X = torch.randn(3, 10)  # 输入数据，3个样本，每个样本10个特征
# 多分类标签，3个样本的4分类标签
y = torch.tensor([[0.0, 1.0, 0.0, 0.0], 
                  [1.0, 0.0, 0.0, 0.0], 
                  [0.0, 0.0, 1.0, 0.0]])

# 3. 实例化模型和损失函数 + 优化器
model = MLP()
loss_fn = nn.CrossEntropyLoss()  # 多分类交叉熵损失函数
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)  # 随机梯度下降优化器

# 4. 训练模型
for epoch in range(100):  # 训练100轮
    y_pred = model(X)  # 前向传播得到预测结果
    loss = loss_fn(y_pred, y)  # 计算损失
    loss.backward()  # 反向传播计算梯度
    optimizer.step()  # 更新模型参数
    optimizer.zero_grad()  # 清零梯度
    if epoch % 10 == 0:  # 每10轮打印一次损失
        print(f'Epoch {epoch}, Loss: {loss.item()}')

Epoch 0, Loss: 1.5318692922592163
Epoch 10, Loss: 1.4704093933105469
Epoch 20, Loss: 1.4166380167007446
Epoch 30, Loss: 1.3693227767944336
Epoch 40, Loss: 1.3267844915390015
Epoch 50, Loss: 1.2873826026916504
Epoch 60, Loss: 1.2504639625549316
Epoch 70, Loss: 1.2159992456436157
Epoch 80, Loss: 1.183521032333374
Epoch 90, Loss: 1.152748942375183


#### 6. Cross Entropy 的直观理解
可以把 Cross Entropy 理解成：

`模型对正确类别的“信心惩罚”。`

如果模型对正确类别：

`概率很高`

损失很小。

如果模型对正确类别：

`概率很低`

损失很大。

#### 7. 为什么分类问题几乎都用 Cross Entropy？
原因有三个：

1️⃣ 与概率模型一致

Cross Entropy 来源于：

最大似然估计 

2️⃣ 梯度稳定

相比 MSE：
* Cross Entropy 梯度更稳定
* 收敛更快

3️⃣ 与 Softmax 完美匹配

Softmax + CrossEntropy

几乎是分类任务的标准组合。